# TurboEdit-style GPS cycle-slip detection and repair

**Course goal:** discover where a dual-frequency GPS phase record has changed
its unknown integer ambiguity; reject bad individual samples; repair only
integer jumps supported by independent combinations.

The original TurboEdit was introduced by Blewitt (1990) for GPS editing.
Our small `dsoclasses` routine implements the **MW + geometry-free**
principle, with a transparent robust history and a **one-epoch look-ahead**
(offline teaching convenience, not an exact reproduction of GIPSY).

Reference: [Blewitt (1990), *An Automatic Editing Algorithm for GPS data*](https://doi.org/10.1029/GL017i003p00199).


In [ ]:
from os.path import join
import numpy as np
import matplotlib.pyplot as plt
from dsoclasses.gnss.cycle_slip_common import (
    teaching_gps_series, gps_series_from_rinex, combinations,
)

data_path = "../../../data"
rinex_path = join(data_path, "DYNG00GRC_R_20240010000_01D_30S_MO.rnx")

# A reproducible 1 Hz example with known ground truth (not a measured orbit).
clean = teaching_gps_series(n=420, seed=2024)
truth = {120: (3, -2), 240: (1, 1), 330: (77, 60)}
corrupted = clean.with_changes(
    slips=truth,
    code_outliers={300: (6., 0.)},    # one bad code observation
    phase_outliers={370: (5., 0.)},   # one bad carrier epoch, NOT a cycle slip
)
t = corrupted.t_s
print(f'GPS L1 wavelength = {clean.wavelength1_m:.6f} m')
print(f'GPS L2 wavelength = {clean.wavelength2_m:.6f} m')
print(f'Wide-lane wavelength = {clean.wavelength_wide_m:.6f} m')
print(f'GF phase = {clean.gf_metres_per_tecu:.6f} m / TECU (after a fixed ambiguity)')
print(f'Simulated phase slips: {truth}; isolated outliers at t = 300, 370 s')


## 1. Measurements and two complementary combinations

Two simultaneous *code* measurements $P_1,P_2$ [metres] and *carrier*
measurements $L_1,L_2$ [cycles], tracked with **the same signal types**
throughout one arc, are enough to detect many cycle slips. We do **not**
need orbit interpolation or a tropospheric model for this first exercise.

The Melbourne–Wübbena wide-lane combination [cycles]:

$$MW = L_1-L_2 - \frac{f_1P_1+f_2P_2}{(f_1+f_2)\lambda_W}, \quad
\lambda_W=\frac{c}{f_1-f_2}. $$

The geometry-free phase combination [metres]:

$$GF=\lambda_1L_1-\lambda_2L_2,\quad \lambda_i=c/f_i.$$

For a genuine jump $(\Delta N_1,\Delta N_2)$, after removing smooth
ionospheric evolution:

$$\Delta MW \approx \Delta N_1-\Delta N_2,\qquad
\Delta GF \approx \lambda_1\Delta N_1-\lambda_2\Delta N_2.$$

**Why both?** Equal integer slips barely affect MW, while particular
large slip pairs nearly cancel in GF. A bad code sample can affect MW
without affecting GF. A single bad phase sample can look like a slip for
one epoch but does *not* persist.


In [ ]:
mw_clean, gf_clean = combinations(clean)
mw_bad, gf_bad = combinations(corrupted)

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, mw_bad, '.', ms=3, label='Observed MW (corrupted)')
ax[0].plot(t, mw_clean, color='tab:green', lw=1.3, label='Uncorrupted truth')
ax[0].set(ylabel='MW [cycles]', title='Raw combinations: slips, noise and isolated errors')
ax[1].plot(t, gf_bad, '.', ms=3, label='Observed GF (corrupted)')
ax[1].plot(t, gf_clean, color='tab:green', lw=1.3, label='Uncorrupted truth')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='GF [m]')
for a in ax:
    for index in truth:
        a.axvline(index, c='tab:red', alpha=.32, lw=1)
    for index in (300, 370):
        a.axvline(index, c='tab:orange', alpha=.5, lw=1, ls='--')
    a.grid(alpha=.25)
    a.legend(loc='best')
plt.tight_layout()
plt.show()
print('Red lines: persistent slips. Dashed orange: isolated code/phase outliers.')


## 2. Compute the two jump estimates using earlier clean epochs

The module uses the median of earlier MW values as a wide-lane baseline,
and extrapolates the recent GF trend to model **smooth ionospheric change**.
Its per-epoch diagnostics are:

* `mw_prediction`, `gf_prediction_m`: what the *current* epoch should have shown;
* `mw_innovation`, `gf_innovation_m`: observed minus predicted;
* `status`: `ok`, `warmup`, `repaired`, `code_outlier`, `phase_outlier`,
  `unresolved` or `gap`.

Red lines in these figures mark injected slip epochs. Practical detection
floors are 0.45 wide-lane cycles and 0.035 metres GF, enlarged when prior
noise warrants; these are teaching values, not GIPSY settings.


In [ ]:
from dsoclasses.gnss.turboedit import turboedit

result = turboedit(corrupted)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, result.mw_innovation, '.', ms=3, label='MW observed - expected')
ax[0].axhline(+.45, color='grey', ls='--', label='nominal floor ±0.45 cycles')
ax[0].axhline(-.45, color='grey', ls='--')
ax[0].set(ylabel='MW innovation [cycles]', title='Two complementary slip signatures')
ax[1].plot(t, result.gf_innovation_m, '.', ms=3, label='GF observed - expected')
ax[1].axhline(+.035, color='grey', ls='--', label='nominal floor ±0.035 m')
ax[1].axhline(-.035, color='grey', ls='--')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='GF innovation [m]')
for a in ax:
    for index in truth:
        a.axvline(index, color='tab:red', alpha=.25)
    a.grid(alpha=.25)
    a.legend()
plt.tight_layout()
plt.show()


## 3. Classify the anomaly, then solve the integer pair

We check the *following* sample: a real slip persists, whereas an isolated
bad datum normally disappears. For a persistent step, search nearby integer
pairs that satisfy **both** $\Delta MW$ and $\Delta GF$. A plausible rounded
answer is not enough: if the pair is ambiguous, mark `unresolved` and start a
new carrier-ambiguity arc. This is safer than inventing an integer repair.

The $(77,60)$ example is instructive: its GF step is nearly zero, but MW
shows a 17-cycle jump. The $(1,1)$ pair does the opposite: zero MW jump,
but a detectable GF step. An isolated code spike affects MW without GF.


In [ ]:
print(' epoch  classification       ΔMW [cy]   ΔGF [m]    ΔN1    ΔN2')
print('-' * 72)
for event in result.events:
    s1 = '-' if event.slip_l1 is None else str(event.slip_l1)
    s2 = '-' if event.slip_l2 is None else str(event.slip_l2)
    print(f'{event.index:5d}  {event.kind:18s} '
          f'{event.mw_step_cycles:+9.3f}  {event.gf_step_m:+9.4f} '
          f'{s1:>6s} {s2:>6s}')

assert [(e.index, e.slip_l1, e.slip_l2) for e in result.slips] == [
    (120, 3, -2), (240, 1, 1), (330, 77, 60)]
assert result.status[300] == 'code_outlier'
assert result.status[370] == 'phase_outlier'
print('Check passed: correct integers recovered; isolated errors were not repaired as slips.')


In [ ]:
# Work out ONE integer repair by hand, using just the plotted MW/GF steps.
event = next(e for e in result.slips if e.index == 120)
wide_lane_integer = round(event.mw_step_cycles)
n2_float = ((event.gf_step_m - clean.wavelength1_m*wide_lane_integer) /
            (clean.wavelength1_m - clean.wavelength2_m))
n1_float = n2_float + wide_lane_integer
print('At epoch 120:')
print(f'  ΔMW = {event.mw_step_cycles:+.3f} cycles -> ΔN1 - ΔN2 = {wide_lane_integer}')
print(f'  ΔGF = {event.gf_step_m:+.4f} metres')
print(f'  estimated (ΔN1, ΔN2) before rounding = ({n1_float:+.3f}, {n2_float:+.3f})')
print(f'  accepted checked integer pair = ({event.slip_l1:+d}, {event.slip_l2:+d})')
print('  Caution: the module searches and validates competing integer pairs;')
print('  merely rounding this demonstration is NOT a reliable repair rule.')


## 4. Compare the repaired phase record with known truth

Repairs are integer **carrier-phase** offsets, never edits to the code
measurement. Isolated bad epochs remain excluded (`NaN`) for later phase
positioning. This clean truth comparison is available only because the
example is simulated.


In [ ]:
# Apply only the accepted integer repairs. Bad epochs are NaN and stay excluded.
corrected_mw = (result.mw_cycles -
                (corrupted.l1_cycles - result.repaired_l1_cycles) +
                (corrupted.l2_cycles - result.repaired_l2_cycles))
corrected_gf = (result.gf_m -
                clean.wavelength1_m * (corrupted.l1_cycles - result.repaired_l1_cycles) +
                clean.wavelength2_m * (corrupted.l2_cycles - result.repaired_l2_cycles))

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, mw_bad - mw_clean, '.', ms=3, label='Before: MW error')
ax[0].plot(t, corrected_mw - mw_clean, '.', ms=2, label='After: MW error')
ax[0].set(ylabel='MW - truth [cycles]', title='Effect of accepted integer repairs')
ax[1].plot(t, gf_bad - gf_clean, '.', ms=3, label='Before: GF error')
ax[1].plot(t, corrected_gf - gf_clean, '.', ms=2, label='After: GF error')
ax[1].set(xlabel='Time since first epoch [s]', ylabel='GF - truth [m]')
for a in ax:
    a.grid(alpha=.25)
    a.legend()
plt.tight_layout()
plt.show()

valid = np.isfinite(result.repaired_l1_cycles)
print('Maximum carrier L1 difference to known clean truth, after repair [cycles]:',
      np.max(np.abs(result.repaired_l1_cycles[valid] - clean.l1_cycles[valid])))
print('Excluded epochs:', [(i, s) for i, s in enumerate(result.status)
                          if s in ('phase_outlier', 'code_outlier', 'unresolved')])


## 5. Real data: DYNG / G21, 2024-01-01

Now run the **same detector on real GPS observations** recorded at DYNG.
We display the algorithm's actual detection results, the corresponding
MW/GF innovations, and *any* accepted repairs. A zero-repair result is
perfectly possible: we must never invent a real cycle slip to make a plot.

Keep one fixed code/carrier tracking descriptor at each frequency: switching
`L1W` to `L1C`, for instance, changes instrumental biases and can mimic
a slip. The reader preserves missing-epoch gaps.

The RINEX is sampled at **30 seconds**. Small slips and rapid ionospheric
changes may be difficult to separate, so these are **exploratory flags**,
not independently verified ground truth. This section can be skipped if
the classroom file is unavailable.


In [ ]:
real = None
try:
    real = gps_series_from_rinex(rinex_path, satellite='G21', max_samples=350)
except (FileNotFoundError, LookupError) as exc:
    print('DYNG RINEX unavailable:', exc)
else:
    print(f'Loaded {len(real.t_s)} real DYNG observations for {real.satellite}')
    print('Fixed code/carrier descriptors:', real.signals)
    print('Median cadence [s]:', np.median(np.diff(real.t_s)))
    real_mw, real_gf = combinations(real)
    real_result = turboedit(real)

    from collections import Counter
    counts = Counter(event.kind for event in real_result.events)
    print('Detected real-data anomalies:', dict(counts))
    print('Accepted integer repairs:', len(real_result.slips))
    print('Epoch   min     type              MW step [cy] GF step [m]  ΔN1  ΔN2')
    print('-'*78)
    for event in real_result.events[:25]:
        n1 = '-' if event.slip_l1 is None else str(event.slip_l1)
        n2 = '-' if event.slip_l2 is None else str(event.slip_l2)
        print(f'{event.index:5d} {event.time_s/60:7.1f}  {event.kind:17s} '
              f'{event.mw_step_cycles:+11.3f} {event.gf_step_m:+11.4f} '
              f'{n1:>4s} {n2:>4s}')
    if len(real_result.events) > 25:
        print(f'... showing first 25 of {len(real_result.events)} events; '
              'full list is in real_result.events')
    if not real_result.slips:
        print('No confidently repairable real slips in the selected window. '
              'This is a valid experimental result.')


## 6. Real detection evidence: MW and geometry-free innovations

The first two panels show the actual observations. The next two panels
show **observed minus predicted** values used to make decisions. Red lines
indicate accepted integer slips; orange lines are other flagged anomalies.

The dashed ±0.45-cycle and ±0.035-metre lines show the *minimum* TurboEdit
teaching thresholds; actual thresholds are enlarged if estimated noise is
higher. An isolated spike is not automatically a persistent slip.


In [ ]:
if real is not None:
    x = real.t_s/60
    fig, ax = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
    ax[0].plot(x, real_mw, 'k.', ms=2)
    ax[0].set(ylabel='MW [cycles]', title='DYNG G21: raw data and TurboEdit decisions')
    ax[1].plot(x, real_gf, 'k.', ms=2)
    ax[1].set(ylabel='GF [m]')
    ax[2].plot(x, real_result.mw_innovation, '.', ms=3, c='tab:blue')
    ax[2].axhline(+.45, ls='--', c='grey')
    ax[2].axhline(-.45, ls='--', c='grey')
    ax[2].set(ylabel='MW innovation [cy]')
    ax[3].plot(x, real_result.gf_innovation_m, '.', ms=3, c='tab:blue')
    ax[3].axhline(+.035, ls='--', c='grey')
    ax[3].axhline(-.035, ls='--', c='grey')
    ax[3].set(xlabel='Minutes after first selected observation',
              ylabel='GF innovation [m]')
    for a in ax:
        for event in real_result.events:
            c = 'tab:red' if event.kind == 'repaired' else 'tab:orange'
            a.axvline(event.time_s/60, color=c, lw=1, alpha=.4)
        a.grid(alpha=.22)
    plt.tight_layout()
    plt.show()


## 7. Real phase repairs: before versus after

We compute MW and GF again, this time from the **repaired carrier phases**.
Only confidently accepted integer slips are removed. Isolated errors and
unresolved epochs remain `NaN`; the estimator does not make up a correction.

If no real slip was detected, the raw and corrected curves will essentially
overlap. In that case, the following controlled experiment supplies a known
slip in an otherwise *real* observation segment.


In [ ]:
def phase_combinations_after_repair(series, estimate):
    # Recalculate both observables with the corrected L1/L2 carrier cycles.
    # Code measurements remain unchanged; excluded epochs produce NaN.
    wide_code = ((series.f1_hz*series.p1_m + series.f2_hz*series.p2_m) /
                 (series.f1_hz+series.f2_hz))
    mw_fixed = (estimate.repaired_l1_cycles - estimate.repaired_l2_cycles -
                wide_code/series.wavelength_wide_m)
    gf_fixed = (series.wavelength1_m*estimate.repaired_l1_cycles -
                series.wavelength2_m*estimate.repaired_l2_cycles)
    return mw_fixed, gf_fixed

if real is not None:
    real_mw_fixed, real_gf_fixed = phase_combinations_after_repair(real, real_result)
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax[0].plot(real.t_s/60, real_mw, '.', ms=3, alpha=.5, label='Original MW')
    ax[0].plot(real.t_s/60, real_mw_fixed, '.', ms=2, label='After repairs')
    ax[0].set(ylabel='MW [cycles]', title='Actual DYNG data: accepted phase repairs only')
    ax[1].plot(real.t_s/60, real_gf, '.', ms=3, alpha=.5, label='Original GF')
    ax[1].plot(real.t_s/60, real_gf_fixed, '.', ms=2, label='After repairs')
    ax[1].set(xlabel='Minutes after first selected observation', ylabel='GF [m]')
    for a in ax:
        for e in real_result.slips:
            a.axvline(e.time_s/60, color='tab:red', alpha=.5)
        a.grid(alpha=.25)
        a.legend()
    plt.tight_layout()
    plt.show()
    print('Real carrier epochs retained:', np.isfinite(real_result.repaired_l1_cycles).sum(),
          'out of', len(real.t_s))
    print('Number of corrected integer slips:', len(real_result.slips))


## 8. Controlled check using **real** DYNG observations

We cannot assume there happens to be a naturally occurring slip in a short
selected window. To demonstrate repair *on real measurement noise*, choose
a locally quiet segment and inject a **known artificial slip** $(+3,-2)$
at one epoch. The original measurements are our reference. This is a
controlled modification of real data, **not** a naturally occurring DYNG
cycle slip. It also lets us check whether the repaired phases return to
the measured baseline.

If no sufficiently quiet, consecutive segment is available, the cell
reports that honestly and does not create a misleading result.


In [ ]:
if real is not None:
    # Choose a previously unflagged epoch with clean history on both sides.
    choices = [i for i in range(30, len(real.t_s)-3)
               if np.all(np.diff(real.t_s[i-10:i+3]) <= 45)
               and all(s == 'ok' for s in real_result.status[i-10:i+3])]
    if not choices:
        print('No uninterrupted, quiet DYNG segment suitable for a controlled '
              'repair example; try another satellite or a longer window.')
    else:
        injected_epoch = choices[len(choices)//2]
        injected = real.with_changes(slips={injected_epoch: (3, -2)})
        injected_result = turboedit(injected)
        injection_events = [e for e in injected_result.events
                            if e.index == injected_epoch]
        found = injection_events[0] if injection_events else None
        print(f'Artificial slip added to real DYNG data at index {injected_epoch}: '
              'L1 +3, L2 -2 cycles')
        print('Detector result:', found if found else 'not flagged')
        print('Accepted exact repair:', bool(found and found.kind == 'repaired'
                                             and (found.slip_l1, found.slip_l2) == (3, -2)))

        injected_mw, injected_gf = combinations(injected)
        injected_mw_fixed, injected_gf_fixed = phase_combinations_after_repair(
            injected, injected_result)
        around = slice(max(0, injected_epoch-12),
                       min(len(real.t_s), injected_epoch+18))
        local_t = (real.t_s[around]-real.t_s[injected_epoch])/60
        fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        ax[0].plot(local_t, (injected_mw-real_mw)[around], '.-',
                   label='Injected MW minus measured baseline')
        ax[0].plot(local_t, (injected_mw_fixed-real_mw)[around], '.-',
                   label='After accepted repairs')
        ax[0].set(ylabel='MW deviation [cycles]',
                  title='Artificial slip on actual DYNG observations')
        ax[1].plot(local_t, (injected_gf-real_gf)[around], '.-',
                   label='Injected GF minus measured baseline')
        ax[1].plot(local_t, (injected_gf_fixed-real_gf)[around], '.-',
                   label='After accepted repairs')
        ax[1].set(xlabel='Minutes relative to injected slip', ylabel='GF deviation [m]')
        for a in ax:
            a.axvline(0, color='tab:red', alpha=.5, ls='--')
            a.grid(alpha=.25)
            a.legend()
        plt.tight_layout()
        plt.show()
        if not found or found.kind != 'repaired':
            print('Repair was not accepted. The raw 30-second observations '
                  'may be too noisy or ionospherically variable here.')
